#Rocket Lab
## Arquitetura Medalhão 
#### Camada Bronze

Usaremos como base os notebooks de exemplo dados em aula (`00_Organizacao_do_Ambiente.ipynb` e `01_Ingestao_Bronze.ipynb`) para realizar a primeira tarefa.

In [0]:
# imports
import requests
from datetime import date, timedelta
from pyspark.sql.functions import current_timestamp


In [0]:
# Estrutura criada no Databricks: workspace > bronze > landing
catalog = "workspace"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

landing_path = f"/Volumes/{catalog}/bronze/landing"

In [0]:
# bronze e volume foram criados manualmente. aqui garantimos os demais schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

DataFrame[]

In [0]:
# mapeamento definido no enunciado
arquivos_bronze = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

# confere se todos os arquivos estão no volume
arquivos_volume = [f.name for f in dbutils.fs.ls(landing_path)]
for arquivo in arquivos_bronze:
    if arquivo in arquivos_volume:
        print(f"Ok {arquivo}")
    else:
        print(f"Faltando {arquivo}")

Ok movies_info_TMDB_IMDB.csv
Ok movies_financials_IMDB_TMDB.csv
Ok movies_metrics_IMDB_TMDB.csv
Ok credits_and_tags_IMDB_TMDB.csv
Ok movies_reviews.csv


In [0]:
for arquivo, tabela in arquivos_bronze.items():
    tabela_completa = f"{bronze_schema}.{tabela}"

    # Se a tabela já existe, não carrega de novo (evita duplicar dados)
    if spark.catalog.tableExists(tabela_completa):
        print(f"{tabela_completa} já existe, pulando")
        continue

    # leitura sem alterar nada: tudo como texto (a limpeza é feita na Silver)
    # multiLine e escape: sinopses e comentários podem ter vírgulas, aspas e quebras de linha
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"{landing_path}/{arquivo}")
    )

    # ingestão e grava em Delta (append)
    df.withColumn("ingestion_datetime", current_timestamp()) \
      .write.format("delta").mode("append") \
      .saveAsTable(tabela_completa)

    print(f"{tabela_completa} criada com {df.count()} linhas")

workspace.bronze.tb_movies_info criada com 106596 linhas
workspace.bronze.tb_movies_financials criada com 106165 linhas
workspace.bronze.tb_movies_metrics criada com 103072 linhas
workspace.bronze.tb_credits_and_tags criada com 105608 linhas
workspace.bronze.tb_movies_reviews criada com 32412 linhas


In [0]:
# Padrão: últimos 7 dias (pois a API não tem cotação em fins de semana e feriados)
hoje = date.today()
dbutils.widgets.text("data_inicio", (hoje - timedelta(days=7)).strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", hoje.strftime("%m-%d-%Y"))

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")
print(data_inicio_formatada, "até", data_fim_formatada)

09-30-2026 até 10-07-2026


In [0]:

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url)
cotacoes = resposta.json()["value"]
print(f"{len(cotacoes)} cotações recebidas")

6 cotações recebidas


In [0]:
tabela_cotacao = f"{bronze_schema}.tb_cotacao_dolar"

df_cotacao = spark.createDataFrame(cotacoes)

# se a tabela já existe, grava só as cotações que ainda não estão nela
if spark.catalog.tableExists(tabela_cotacao):
    df_existente = spark.table(tabela_cotacao).select("dataHoraCotacao")
    df_cotacao = df_cotacao.join(df_existente, on="dataHoraCotacao", how="left_anti")

df_cotacao.withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(tabela_cotacao)

print(f"{df_cotacao.count()} cotações novas gravadas")

6 cotações novas gravadas


In [0]:
for tabela in list(arquivos_bronze.values()) + ["tb_cotacao_dolar"]:
    qtd = spark.table(f"{bronze_schema}.{tabela}").count()
    print(f"{tabela}: {qtd} linhas")

display(spark.table(f"{bronze_schema}.tb_movies_info").limit(5))
display(spark.table(f"{bronze_schema}.tb_cotacao_dolar"))

tb_movies_info: 106596 linhas
tb_movies_financials: 106165 linhas
tb_movies_metrics: 103072 linhas
tb_credits_and_tags: 105608 linhas
tb_movies_reviews: 32412 linhas
tb_cotacao_dolar: 6 linhas


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T21:36:07.023Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T21:36:07.023Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T21:36:07.023Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T21:36:07.023Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T21:36:07.023Z


cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.1803,2026-09-30 13:11:44.783262,2026-10-07T21:36:56.846Z
5.2073,2026-10-01 13:10:35.4469,2026-10-07T21:36:56.846Z
5.2232,2026-10-02 13:03:16.256632,2026-10-07T21:36:56.846Z
4.9853,2026-10-05 13:07:36.558602,2026-10-07T21:36:56.846Z
4.9692,2026-10-06 13:03:21.267287,2026-10-07T21:36:56.846Z
4.9929,2026-10-07 13:05:23.249063,2026-10-07T21:36:56.846Z
